[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/02_preprocessing.ipynb)

In [ ]:
import os, sys
from pathlib import Path

# ── Detectar si estamos en Google Colab ──────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Estrategia 1: acceso por ID de carpeta raíz del proyecto
    _p1 = "/content/drive/.shortcut-targets-by-id/13VEeA8Jt0G_NNpJelWZrhacBhA_mRpnU"
    # Estrategia 2: acceso por ID de TrainData (subir un nivel)
    _p2 = "/content/drive/.shortcut-targets-by-id/13Dj5DYGOwgMGomMC8gCySKGr0ne8hOpU"

    if os.path.exists(_p1) and os.path.isdir(_p1):
        DATA_ROOT_OVERRIDE = _p1
        print(f"Estrategia 1 OK: {_p1}")
    elif os.path.exists(_p2) and os.path.isdir(_p2):
        DATA_ROOT_OVERRIDE = str(Path(_p2).parent)
        print(f"Estrategia 2 OK: {DATA_ROOT_OVERRIDE}")
    else:
        import subprocess
        _r = subprocess.run(
            ["find", "/content/drive/MyDrive", "-name", "TrainData",
             "-type", "d", "-maxdepth", "6"],
            capture_output=True, text=True, timeout=30
        )
        _hits = [p.replace('/TrainData', '') for p in _r.stdout.strip().splitlines() if p]
        if _hits:
            DATA_ROOT_OVERRIDE = _hits[0]
            print(f"Estrategia 3 OK: {DATA_ROOT_OVERRIDE}")
        else:
            print("No se encontró el dataset. Ajusta DATA_ROOT_OVERRIDE manualmente:")
            print("  DATA_ROOT_OVERRIDE = 'ruta/en/tu/Drive'")

    if DATA_ROOT_OVERRIDE:
        print(f"Dataset: {DATA_ROOT_OVERRIDE}")
else:
    print('Entorno local — se usará detección automática de rutas.')


# 02 — Preprocesamiento y Data Augmentation

Objetivo: definir la pipeline de normalización y augmentación geométrica/espectral sobre parches de 14 canales.

---

## 1. Imports

In [ ]:
import os, sys
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL DATASET
# Si la celda de Drive de arriba encontró el dataset, DATA_ROOT_OVERRIDE
# ya está definida. Si no, puedes ajustarla manualmente aquí:
# DATA_ROOT_OVERRIDE = "/content/drive/MyDrive/landslide4sense"
# ══════════════════════════════════════════════════════════════════════════════

# Preservar DATA_ROOT_OVERRIDE si ya fue fijada por la celda de Drive
try:
    DATA_ROOT_OVERRIDE
except NameError:
    DATA_ROOT_OVERRIDE = None

# ── Detección automática ──────────────────────────────────────────────────────
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
    print(f"Usando ruta configurada: {DATA_ROOT}")
else:
    _cwd = Path(os.getcwd())
    _candidates = [
        _cwd / ".." / "data",
        _cwd / "..",
        _cwd / "data",
        _cwd,
        Path("/content/drive/MyDrive/landslide4sense"),
        Path("/content/drive/MyDrive/Landslide_ML"),
        Path("/content/landslide4sense"),
        Path("/content/data"),
        Path("/content"),
    ]
    DATA_ROOT = None
    for _c in _candidates:
        if (_c / "TrainData" / "img").exists():
            DATA_ROOT = _c.resolve()
            print(f"Dataset detectado automáticamente en: {DATA_ROOT}")
            break

if DATA_ROOT is None or not (DATA_ROOT / "TrainData" / "img").exists():
    print("""
⚠️  Dataset no encontrado. Opciones:

━━━  OPCIÓN A — Google Drive (recomendado)  ━━━
  Ejecuta la celda de Drive de arriba y vuelve a correr esta celda.

━━━  OPCIÓN B — Kaggle API  ━━━
  !pip install kaggle -q
  # Sube tu kaggle.json y ejecuta:
  !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
  !kaggle datasets download -d landslide4sense/competition -p /content/
  !unzip -q /content/competition.zip -d /content/landslide4sense/
  DATA_ROOT_OVERRIDE = "/content/landslide4sense"
""")
    raise FileNotFoundError("Dataset no encontrado. Establece DATA_ROOT_OVERRIDE con la ruta correcta.")

# ── Verificar estructura del dataset ─────────────────────────────────────────
n_train = len(list((DATA_ROOT / "TrainData" / "img").glob("*.h5")))
print(f"✓ Dataset encontrado en: {DATA_ROOT}")
print(f"  TrainData/img : {n_train} archivos .h5")


## 2. Cargar un parche de ejemplo

In [ ]:
img_dir  = DATA_ROOT / "TrainData" / "img"
mask_dir = DATA_ROOT / "TrainData" / "mask"

files = sorted(img_dir.glob("*.h5"))
# Buscar un parche con deslizamiento
for f in files:
    mf = mask_dir / f.name
    with h5py.File(mf,"r") as hf:
        mask = hf[list(hf.keys())[0]][()]
    if mask.max() > 0:
        with h5py.File(f,"r") as hf:
            patch = hf[list(hf.keys())[0]][()].astype("float32")
        break

print(f"Parche: {f.name}  shape={patch.shape}  dtype={patch.dtype}")
print(f"Rango bruto: [{patch.min():.4f}, {patch.max():.4f}]")
print(f"Deslizamiento — área: {mask.mean()*100:.2f}%")

## 3. Estrategias de normalización

In [ ]:
from src.dataset import normalize_patch, minmax_patch

patch_z    = normalize_patch(patch)      # Z-score por canal
patch_mm   = minmax_patch(patch)         # Min-max [0,1] por canal

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
ch_idx = [2, 3, 9]  # RGB, NIR, DEM
labels = ["Bruto", "Z-score", "Min-max"]
data   = [patch, patch_z, patch_mm]

for row, (d, lbl) in enumerate([(patch, "Bruto"), (patch_z, "Z-score"), (patch_mm, "Min-max")]):
    for col, ch in enumerate(ch_idx):
        ax = axes[col][row] if row < 2 else None
    pass

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for col, ch in enumerate(ch_idx):
    for row, (d, lbl) in enumerate(zip(data, labels)):
        axes[row][col].hist(d[:,:,ch].flatten(), bins=60, color="#3498db", alpha=0.8)
        axes[row][col].set_title(f"{lbl} — {CHANNEL_NAMES[ch]}")
        axes[row][col].axvline(d[:,:,ch].mean(), color="red", ls="--", lw=1.5, label=f"μ={d[:,:,ch].mean():.2f}")
        axes[row][col].legend(fontsize=9)
plt.suptitle("Distribución por canal tras distintas normalizaciones", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("Estrategia seleccionada: Z-score por canal (per_channel)")

## 4. Data Augmentation — visualización

In [ ]:
cfg = get_debug_config()
aug = Augmenter(cfg)

n_aug = 6
fig, axes = plt.subplots(2, n_aug, figsize=(3*n_aug, 6))

axes[0][0].set_title("Original")
rgb_orig = patch[:,:,[2,1,0]]
rgb_disp = ((rgb_orig - rgb_orig.min()) / (rgb_orig.ptp()+1e-8)).clip(0,1)
axes[0][0].imshow(rgb_disp)
axes[1][0].imshow(mask, cmap="Reds")

for i in range(1, n_aug):
    p_aug, m_aug = aug(patch.copy(), mask.copy())
    rgb_a = p_aug[:,:,[2,1,0]]
    rgb_a = ((rgb_a - rgb_a.min()) / (rgb_a.ptp()+1e-8)).clip(0,1)
    axes[0][i].imshow(rgb_a)
    axes[0][i].set_title(f"Aug #{i}")
    axes[1][i].imshow(m_aug, cmap="Reds")

for ax in axes.flat:
    ax.axis("off")

plt.suptitle("Data Augmentation — RGB (fila 1) y Máscara (fila 2)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Dataset con DataLoader — test de forma

In [ ]:
import torch
from torch.utils.data import DataLoader

ds = Landslide4SenseDataset(
    img_dir=str(img_dir), mask_dir=str(mask_dir),
    indices=list(range(32)), transform=aug, normalize=True, task="classification"
)
loader = DataLoader(ds, batch_size=8, shuffle=True, num_workers=0)

batch = next(iter(loader))
print(f"Forma image: {batch['image'].shape}   (B, C, H, W)")
print(f"Forma label: {batch['label'].shape}   (B,)")
print(f"dtype:       {batch['image'].dtype}")
print(f"Rango tras normalización: [{batch['image'].min():.3f}, {batch['image'].max():.3f}]")
print("\n✓ Pipeline de preprocesamiento validada.")